## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv() ## aloading all the environment variable
 
groq_api_key=os.getenv("GROQ_API_KEY")


In [2]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.3-70b-versatile",groq_api_key=groq_api_key)
model

e:\Desktop\GenAI\GenAI\ChatbotsWithConvHistory\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000242D9F67400>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000242D9F67AF0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [3]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi , My name is Krish and I am a Chief AI Engineer")])

AIMessage(content="Nice to meet you, Krish. It's great to connect with a Chief AI Engineer. That sounds like a fascinating and challenging role. What kind of projects are you currently working on, and what aspects of AI engineering are you most passionate about?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 48, 'total_tokens': 98, 'completion_time': 0.154012066, 'completion_tokens_details': None, 'prompt_time': 0.004717645, 'prompt_tokens_details': None, 'queue_time': 0.051082834, 'total_time': 0.158729711}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f1e83-66a8-7e22-ba24-ff2d6344bf9c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 50, 'total_tokens': 98})

In [4]:
from langchain_core.messages import AIMessage
#remembers the previous context if we give conversation as a list of messages and ask a new question.
model.invoke(
    [
        HumanMessage(content="Hi , My name is Krish and I am a Chief AI Engineer"),
        AIMessage(content="Hello Krish! It's nice to meet you. \n\nAs a Chief AI Engineer, what kind of projects are you working on these days? \n\nI'm always eager to learn more about the exciting work being done in the field of AI.\n"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content="Your name is Krish, and you're a Chief AI Engineer.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 117, 'total_tokens': 131, 'completion_time': 0.028892936, 'completion_tokens_details': None, 'prompt_time': 0.005333765, 'prompt_tokens_details': None, 'queue_time': 0.162562965, 'total_time': 0.034226701}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f1e83-68ea-7112-9259-4c789c6f9b27-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 117, 'output_tokens': 14, 'total_tokens': 131})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [5]:
from langchain_community.chat_message_histories import ChatMessageHistory #Imports ChatMessageHistory, which stores all messages (user + AI) of a conversation.
from langchain_core.chat_history import BaseChatMessageHistory #Imports the base class used as the return type for chat history objects.

from langchain_core.runnables.history import RunnableWithMessageHistory #Imports a wrapper that automatically adds chat history to your LLM.

store={}

#ensures that when multiple users interact with the same LLM, different sessions are maintained. #Defines a function that takes a session ID and returns that session's chat history.
def get_session_history(session_id:str)->BaseChatMessageHistory:#BaseChatMessageHistory is the return type. LangChain uses the parent class (BaseChatMessageHistory) to allow any child class (ChatMessageHistory, Redis, MongoDB, etc.) to be returned. This is a common object-oriented programming practice called polymorphism.
    if session_id not in store:#Checks if this session ID already has a chat history.
        store[session_id]=ChatMessageHistory() #If not, creates a new empty chat history and stores it using the session ID.
    return store[session_id] #Returns the chat history for that session.

#Wraps the LLM so it automatically loads and updates the correct chat history for each session.
with_message_history=RunnableWithMessageHistory(model,get_session_history)

C:\Users\naman\AppData\Local\Temp\ipykernel_76652\3036856667.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory #Imports ChatMessageHistory, which stores all messages (user + AI) of a conversation.
e:\Desktop\GenAI\GenAI\ChatbotsWithConvHistory\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [6]:
config={"configurable":{"session_id":"chat1"}}#hardcoding it for example.

In [7]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Krish and I am a Chief AI Engineer")],
    config=config
)

In [8]:
response.content

"Nice to meet you, Krish! As a Chief AI Engineer, you must be working on some fascinating projects, leveraging the power of artificial intelligence to drive innovation and solve complex problems. What specific areas of AI do you specialize in, and what are some of the most exciting projects you're currently working on? I'm all ears!"

In [9]:
#Keeping the session id same
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

AIMessage(content="Your name is Krish, and you're a Chief AI Engineer.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 129, 'total_tokens': 143, 'completion_time': 0.019211982, 'completion_tokens_details': None, 'prompt_time': 0.012067845, 'prompt_tokens_details': None, 'queue_time': 0.166094254, 'total_time': 0.031279827}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f1e83-6d72-76f2-8cf3-73b89c93939f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 129, 'output_tokens': 14, 'total_tokens': 143})

In [10]:
## change the config-->session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

"I don't know your name. I'm a large language model, I don't have the ability to know your personal information or recall previous conversations. Each time you interact with me, it's a new conversation. If you'd like to share your name, I'd be happy to chat with you!"

In [11]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config1
)
response.content

"Nice to meet you, John! It's great to have a name to associate with our conversation. How's your day going so far? Is there something I can help you with or would you like to just chat?"

In [12]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'I remember! Your name is John. We just established that a minute ago.'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [13]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Answer all the question to the nest of your ability"),
        MessagesPlaceholder(variable_name="messages")#Earlier we were directly giving human messaages but now we will use a placeholder whose key name is messages and will give human messages in that as a list while invoking the chain.
    ]
)

chain=prompt|model

In [14]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Krish")]})

AIMessage(content="Hello Krish, it's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 56, 'total_tokens': 82, 'completion_time': 0.071503658, 'completion_tokens_details': None, 'prompt_time': 0.013154282, 'prompt_tokens_details': None, 'queue_time': 0.051203684, 'total_time': 0.08465794}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f1e83-7416-7573-b24a-8428e1b2935a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 56, 'output_tokens': 26, 'total_tokens': 82})

In [15]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

e:\Desktop\GenAI\GenAI\ChatbotsWithConvHistory\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [16]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Krish")],
    config=config
)

response

AIMessage(content="Hello Krish, it's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 56, 'total_tokens': 82, 'completion_time': 0.074648916, 'completion_tokens_details': None, 'prompt_time': 0.00269769, 'prompt_tokens_details': None, 'queue_time': 0.05155116, 'total_time': 0.077346606}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f1e83-75bb-7ca3-a9fc-63e745148bc1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 56, 'output_tokens': 26, 'total_tokens': 82})

In [17]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is Krish.'

In [18]:
## Add more complexity (multiple input variables)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [19]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Krish")],"language":"Hindi"})
response.content

'नमस्ते कृष, मैं आपका सहायक हूँ। मैं आपकी किस प्रकार से सहायता कर सकता हूँ?'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to combine new messages with the chat history.

In [20]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages" #input_messages_key tells RunnableWithMessageHistory which key in your input dictionary contains the current chat messages.
)

# Here, "messages" means that when you invoke the chain, your input should look like:

# with_message_history.invoke(
#     {
#         "messages": [
#             HumanMessage(content="Hi!")
#         ]
#     },
#     config={"configurable": {"session_id": "abc123"}}
# )

# RunnableWithMessageHistory will:

# Look inside the "messages" key.
# Fetch the previous conversation from get_session_history(session_id).
# Combine the old messages with the new messages.
# Pass the combined message list to chain.
# Save the new conversation back into the history.

e:\Desktop\GenAI\GenAI\ChatbotsWithConvHistory\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [21]:
config = {"configurable": {"session_id": "chat4"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Krish")],"language":"Hindi"},
    config=config
)
repsonse.content

'नमस्ते कृष, मैं आपकी कैसे मदद कर सकता हूँ?'

In [22]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config,
)

In [23]:
response.content

'आपका नाम कृष है।'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
trim_messages is used to limit the chat history sent to the LLM by removing older messages when the conversation becomes too long. This helps keep the prompt within the model's context window, reduces token usage and cost, and ensures the model focuses on the most recent or most relevant parts of the conversation.

In [24]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,#Keeps the total message history within approximately 45 tokens.
    strategy="last",#Preserves the most recent messages and removes the oldest ones first.
    token_counter=model,#Uses the model's tokenizer to accurately count the number of tokens.
    include_system=True,#Always keeps the SystemMessage even if other messages are removed.
    allow_partial=False,#Keeps or removes a message as a whole; it never cuts a message in the middle.
    start_on="human"#Ensures the trimmed conversation starts with a HumanMessage, avoiding an invalid history that begins with an AI response.
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

e:\Desktop\GenAI\GenAI\ChatbotsWithConvHistory\venv\lib\site-packages\langchain_core\language_models\base.py:447: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))
e:\Desktop\GenAI\GenAI\ChatbotsWithConvHistory\venv\lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\naman\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Develo

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [ ]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

#prompt expects its input in a specific dictionary format, while trimmer only returns a list of messages. If you do this:
# chain = trimmer | prompt | model
# Input:
# {
#     "messages": messages,
#     "language": "English"
# }
#trimmer takes the entire input and returns only the trimmed messages: 
#[
#     SystemMessage(...),
#     HumanMessage(...),
#     AIMessage(...)
# ]
#Notice that the dictionary is gone. The "language" key is lost.

# So ChatPromptTemplate expects
# {
#     "messages": [...],
#     "language": "English"
# }

# But instead it receives
# [
#     SystemMessage(...),
#     HumanMessage(...)
# ] It has no "messages" key and no "language" key, so it fails.

# The assign() below puts the trimmed messages list back into the original dictionary:
chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)#first get all messages using itemgetter and then apply trimmer
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

"I don't know what your favorite ice cream is, as we just started chatting and I don't have any information about your preferences. Would you like to tell me?"

In [26]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'You asked "what\'s 2 + 2" and the answer was 4.'

In [ ]:
## Lets wrap this in the MEssage History 
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

e:\Desktop\GenAI\GenAI\ChatbotsWithConvHistory\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [28]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

"I don't know your name, you haven't told me yet. Would you like to share it with me?"

In [29]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

'You didn\'t ask a math problem. This conversation just started, and your first message was "you\'re a good assistant". If you\'d like to ask a math problem or any other question, I\'m here to help!'